In [1]:
import numpy as np
import time
from pynq import Overlay, allocate
import cv2

# 1. 하드웨어 초기화
overlay = Overlay("design_1.bit")
dma = overlay.axi_dma_0
accel = overlay.myip_0

# 레지스터 주소 정의
REG_CONFIG  = 0x00
REG_SIZE    = 0x04
REG_COEFF_R = [0x0C, 0x10, 0x14]
REG_COEFF_G = [0x18, 0x1C, 0x20]
REG_COEFF_B = [0x24, 0x28, 0x2C]

In [18]:
def pack_coeffs(c1, c2, c3):
    return ((c1 & 0xFF) << 16) | ((c2 & 0xFF) << 8) | (c3 & 0xFF)

# 테스트할 해상도 리스트 (Width, Height)
resolutions = [
    (32, 32),
    (64, 64),
    (128, 128),
    (256, 256),
    (640, 480), # VGA
    (1280, 720), # HD
    (1920, 1080), # FHD
    (1440, 2560), # QHD
]

# 필터 계수 설정 (Sharpening)
coeffs = [[0, -1, 0], [-1, 5, -1], [0, -1, 0]]
kernel = np.array(coeffs, dtype=np.int8)

print(f"{'Resolution':<15} | {'setup (s)':<12} | {'HW Time (s)':<12} | {'CV Time (s)':<12} | {'Speedup':<8}")
print("-" * 80)

for width, height in resolutions:
    # 2. 더미 데이터 생성 (RGBA 채널 4)
    # 실제 파일 로드 대신 random 데이터를 사용하여 순수 연산/전송 속도만 측정
    test_data = np.random.randint(0, 256, (height, width, 4), dtype=np.uint8)
    
    in_buffer = allocate(shape=(height, width, 4), dtype=np.uint8)
    out_buffer = allocate(shape=(height, width, 4), dtype=np.uint8)
    in_buffer[:] = test_data

    # --- HW 처리 ---
    # IP 설정
    hw_setup = time.time()
    accel.write(REG_CONFIG, (1 << 4)) # Reset
    accel.write(REG_SIZE, ((height - 1) << 16) | (width - 1))
    for i in range(3):
        val = pack_coeffs(coeffs[i][0], coeffs[i][1], coeffs[i][2])
        accel.write(REG_COEFF_R[i], val)
        accel.write(REG_COEFF_G[i], val)
        accel.write(REG_COEFF_B[i], val)
    accel.write(REG_CONFIG, (1 << 0)) # Enable

    hw_start = time.time()
    dma.recvchannel.transfer(out_buffer)
    dma.sendchannel.transfer(in_buffer)
    dma.sendchannel.wait()
    dma.recvchannel.wait()
    
    hw_duration = time.time() - hw_start
    hw_setting = hw_start - hw_setup
    
    # --- SW (OpenCV) 처리 ---
    sw_start = time.time()
    # OpenCV는 보통 BGR/RGB 3채널 기준이므로 4채널 처리를 위해 slicing 하거나 그대로 수행
    # 가속기와 동일한 조건(4채널)으로 수행
    cv2.filter2D(test_data, -1, kernel)
    sw_duration = time.time() - sw_start

    speedup = sw_duration / hw_duration
    speedup_2 = sw_duration / (hw_duration + hw_setting)
    print(f"{width:6}  x{height:6} | {hw_setting:<12.5f} | {hw_duration:<12.5f} | {sw_duration:<12.5f} | {speedup:<05.2f}x | {speedup_2:<05.2f}x")

    # 버퍼 해제 (메모리 부족 방지)
    in_buffer.freebuffer()
    out_buffer.freebuffer()
    del in_buffer, out_buffer

print("-" * 60)
print("Benchmark Completed.")

Resolution      | setup (s)    | HW Time (s)  | CV Time (s)  | Speedup 
--------------------------------------------------------------------------------
    32  x    32 | 0.00039      | 0.00106      | 0.00053      | 0.510x | 0.370x
    64  x    64 | 0.00043      | 0.00102      | 0.00164      | 1.600x | 1.130x
   128  x   128 | 0.00038      | 0.00105      | 0.00597      | 5.710x | 4.180x
   256  x   256 | 0.00038      | 0.00151      | 0.02327      | 15.37x | 12.26x
   640  x   480 | 0.00040      | 0.00344      | 0.11440      | 33.23x | 29.75x
  1280  x   720 | 0.00041      | 0.00847      | 0.34732      | 41.01x | 39.13x
  1920  x  1080 | 0.00041      | 0.01779      | 0.78593      | 44.18x | 43.19x
  1440  x  2560 | 0.00040      | 0.03083      | 1.38978      | 45.08x | 44.50x
------------------------------------------------------------
Benchmark Completed.
